# Week 73, Day 2: pyproject.toml

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w073d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import tomllib

text = '''
[project]
name = "textkit"
version = "0.3.0"
dependencies = ["regex>=2024.1", "rich"]

[project.optional-dependencies]
dev = ["pytest>=8", "mypy"]

[project.scripts]
textkit = "textkit.cli:main"
'''

data = tomllib.loads(text)
print(data["project"]["name"], data["project"]["version"])
print(len(data["project"]["dependencies"]), data["project"]["optional-dependencies"]["dev"])
print(data["project"]["scripts"])

In [ ]:
import re, tomllib

def normalise(name):
    return re.sub(r"[-_.]+", "-", name).lower()

print([normalise(n) for n in ["Text_Kit", "text.kit", "TEXT--kit", "textkit"]])

pyproject = tomllib.loads('''
[project]
name = "Text_Kit"
dynamic = ["version"]
requires-python = ">=3.10"
dependencies = ["rich>=13,<15", "tomli>=2; python_version < '3.11'"]
''')
project = pyproject["project"]
print(normalise(project["name"]), "version" in project.get("dynamic", []))
for req in project["dependencies"]:
    spec, _, marker = req.partition(";")
    print(f"{spec.strip():<16} marker: {marker.strip() or '(none)'}")

> **Key idea.** pyproject.toml is TOML: quoted strings, [tables], arrays and inline tables, read with tomllib (binary files). [build-system] names the packages needed to build and the backend that builds. [project] holds name and version (or lists them in dynamic), description, readme, requires-python, licence, dependencies, optional-dependencies (extras), scripts (console commands) and urls; [tool.X] belongs to tools. Names compare after normalising: lower case, runs of - _ . become -. Requirement strings give a name, extras, version clauses and an optional environment marker after ;.

### Exercise: Reading a pyproject.toml

Write `summary(text)`: parse `text` with `tomllib` and return a dict with `"name"`, `"version"` (or `"dynamic"` if `version` is listed in `project.dynamic`), `"python"` (the `requires-python` string, or `"any"`), `"dependencies"` (the number of them), `"extras"` (the sorted names of the optional-dependency groups) and `"commands"` (the sorted names of the console scripts). A missing `[project]` table or a missing name raises `ValueError`; so does a missing version that isn't dynamic.

In [ ]:
import tomllib

def summary(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_summary():\n    \"reading the project table\"\n    text = '[project]\\nname = \"textkit\"\\nversion = \"0.3.0\"\\nrequires-python = \">=3.10\"\\ndependencies = [\"rich\", \"regex\"]\\n\\n[project.optional-dependencies]\\npdf = [\"pypdf\"]\\ndev = [\"pytest\"]\\n\\n[project.scripts]\\ntk = \"textkit.cli:main\"\\ntextkit = \"textkit.cli:main\"\\n'\n    assert summary(text) == {\"name\": \"textkit\", \"version\": \"0.3.0\", \"python\": \">=3.10\", \"dependencies\": 2, \"extras\": [\"dev\", \"pdf\"], \"commands\": [\"textkit\", \"tk\"]}\n    assert summary('[project]\\nname = \"x\"\\ndynamic = [\"version\"]\\n') == {\"name\": \"x\", \"version\": \"dynamic\", \"python\": \"any\", \"dependencies\": 0, \"extras\": [], \"commands\": []}\n    for bad in ['[tool.x]\\na = 1\\n', '[project]\\nversion = \"1\"\\n', '[project]\\nname = \"x\"\\n']:\n        try:\n            summary(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Project names

Write `valid_name(name)`: true if the name has only letters, digits, `.`, `_` and `-`, and starts and ends with a letter or digit (a single letter or digit is fine). Then `normalise(name)` (PEP 503) and `same_project(a, b)`. Finally `clashes(names)`: given a list of names, return the sorted list of normalised names that more than one entry shares.

In [ ]:
import re

def valid_name(name):
    ___

def normalise(name):
    ___

def same_project(a, b):
    ___

def clashes(names):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"valid, normalised and clashing names\"\n    assert all(valid_name(n) for n in [\"textkit\", \"Text_Kit\", \"a\", \"zope.interface\", \"py-2\"])\n    assert not any(valid_name(n) for n in [\"\", \"-kit\", \"kit-\", \"text kit\", \"kit!\", \"_x\"])\n    assert normalise(\"Zope.Interface\") == \"zope-interface\" and normalise(\"a__b--c..d\") == \"a-b-c-d\"\n    assert same_project(\"Text_Kit\", \"text.kit\") and not same_project(\"textkit\", \"text-kit\")\n    assert clashes([\"Text_Kit\", \"text-kit\", \"other\", \"TEXT.KIT\", \"Other\", \"unique\"]) == [\"other\", \"text-kit\"]")

### Exercise: Parsing requirements

Write `parse_requirement(text)`, returning a dict with `"name"` (normalised), `"extras"` (a sorted list), `"specifiers"` (a list of `(operator, version)` pairs in the order written) and `"marker"` (the text after `;`, stripped, or `None`). Operators are `==`, `!=`, `<=`, `>=`, `<`, `>`, `~=` and `===`; spaces may appear anywhere around them. A missing or invalid name, or a clause without a known operator, raises `ValueError`.

In [ ]:
import re

def parse_requirement(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"names, extras, specifiers and markers\"\n    assert parse_requirement(\"requests\") == {\"name\": \"requests\", \"extras\": [], \"specifiers\": [], \"marker\": None}\n    assert parse_requirement(\"Rich >= 13, <15\") == {\"name\": \"rich\", \"extras\": [], \"specifiers\": [(\">=\", \"13\"), (\"<\", \"15\")], \"marker\": None}\n    r = parse_requirement(\"Text_Kit[pdf, dev]~=0.3 ; python_version < '3.11'\")\n    assert r == {\"name\": \"text-kit\", \"extras\": [\"dev\", \"pdf\"], \"specifiers\": [(\"~=\", \"0.3\")], \"marker\": \"python_version < '3.11'\"}\n    assert parse_requirement(\"django==5.0.*\")[\"specifiers\"] == [(\"==\", \"5.0.*\")]\n    for bad in [\"\", \">=1.0\", \"rich >> 2\", \"rich[pdf\"]:\n        try:\n            parse_requirement(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Console scripts

Write `parse_scripts(project)`: `project` is the parsed `[project]` table. Return a dict of command name to `(module, function)` from its `scripts`, checking each value has the form `module:function`, where the module is a dotted name of identifiers and the function an identifier (or a dotted attribute path such as `Cli.run`). An invalid value raises `ValueError` naming the command. Then `wrapper(module, function)`: the text of the tiny program an installer writes for a console script: exactly `f"import sys\nfrom {module} import {first}\nsys.exit({function}())\n"`, where `first` is the first part of `function`.

In [ ]:
def parse_scripts(project):
    ___

def wrapper(module, function):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scripts():\n    \"entry points\"\n    project = {\"scripts\": {\"textkit\": \"textkit.cli:main\", \"tk-admin\": \"textkit.admin.tools:Admin.run\"}}\n    assert parse_scripts(project) == {\"textkit\": (\"textkit.cli\", \"main\"), \"tk-admin\": (\"textkit.admin.tools\", \"Admin.run\")}\n    assert parse_scripts({}) == {}\n    for bad in [\"textkit.cli\", \"textkit cli:main\", \"textkit.cli:\", \":main\", \"9kit:main\"]:\n        try:\n            parse_scripts({\"scripts\": {\"x\": bad}})\n            assert False, bad\n        except ValueError as e:\n            assert \"x\" in str(e)\n    assert wrapper(\"textkit.cli\", \"main\") == \"import sys\\nfrom textkit.cli import main\\nsys.exit(main())\\n\"\n    assert wrapper(\"t.a\", \"Admin.run\") == \"import sys\\nfrom t.a import Admin\\nsys.exit(Admin.run())\\n\"")

### Exercise: Writing TOML

`tomllib` can't write. Write `to_toml(data)` for the shapes a `pyproject.toml` needs: a dict whose values are dicts (tables, possibly nested, written with dotted headers such as `[project.urls]`). Inside a table, values are strings (written with double quotes, escaping `\` and `"`), integers, booleans (`true`/`false`) or lists of strings (on one line). Write each table's own simple values first, then its sub-tables, with a blank line between tables, and keys in the order given. Skip a header for a table that has only sub-tables. The result must parse back to the same data with `tomllib.loads`.

In [ ]:
import tomllib

def to_toml(data):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_writer():\n    \"a round trip through tomllib\"\n    import tomllib\n    data = {\n        \"build-system\": {\"requires\": [\"hatchling\"], \"build-backend\": \"hatchling.build\"},\n        \"project\": {\n            \"name\": \"textkit\", \"version\": \"0.3.0\", \"dependencies\": [\"rich>=13\", 'tomli; python_version < \"3.11\"'],\n            \"urls\": {\"Homepage\": \"https://example.com\"},\n            \"optional-dependencies\": {\"dev\": [\"pytest\"]},\n        },\n        \"tool\": {\"pytest\": {\"ini_options\": {\"testpaths\": [\"tests\"], \"strict\": True, \"level\": 2}}},\n    }\n    text = to_toml(data)\n    assert tomllib.loads(text) == data\n    assert text.startswith('[build-system]\\nrequires = [\"hatchling\"]\\nbuild-backend = \"hatchling.build\"\\n\\n[project]\\n')\n    assert \"[tool]\" not in text and \"[tool.pytest]\" not in text and \"[tool.pytest.ini_options]\" in text\n    assert tomllib.loads(to_toml({\"t\": {\"path\": 'C:\\\\new \"dir\"'}})) == {\"t\": {\"path\": 'C:\\\\new \"dir\"'}}")

### Exercise: Which Pythons are supported?

Write `allows(requires, version)`: `requires` is a `requires-python` string made of comma-separated clauses with the operators `>=`, `>`, `<=`, `<`, `==` and `!=` (no wildcards), and `version` a string such as `"3.12.1"`. Compare versions as tuples of integers, padding the shorter one with zeros (so `3.12` equals `3.12.0`). Return whether every clause holds. An empty `requires` allows everything. Then `supported(requires, versions)`: the versions in the list that are allowed, in order.

In [ ]:
def allows(requires, version):
    ___

def supported(requires, versions):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_python():\n    \"requires-python\"\n    assert allows(\">=3.10\", \"3.12.1\") and not allows(\">=3.10\", \"3.9.18\")\n    assert allows(\">=3.10, <4\", \"3.13\") and not allows(\">=3.10,<3.12\", \"3.12.0\")\n    assert allows(\"\", \"2.7\") and allows(\"!=3.11.2\", \"3.11.3\") and not allows(\"!=3.11.2\", \"3.11.2\")\n    assert allows(\"==3.12\", \"3.12.0\") and allows(\"<=3.12\", \"3.12\") and not allows(\">3.12\", \"3.12.0\")\n    assert supported(\">=3.9,!=3.10.0,<3.13\", [\"3.8\", \"3.9\", \"3.10.0\", \"3.10.1\", \"3.12.9\", \"3.13.0\"]) == [\"3.9\", \"3.10.1\", \"3.12.9\"]")

### Bug hunt: A version that isn't a string

`version = 1.0.0` isn't valid TOML (a version is a **string**), so `tomllib` raises `TOMLDecodeError`. Fix the TOML so this prints `textkit 1.0.0`.

In [ ]:
import tomllib

text = '''
[project]
name = "textkit"
version = 1.0.0
'''
project = tomllib.loads(text)["project"]
print(project["name"], project["version"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a quoted version\"\n    expect_output(\"textkit 1.0.0\")")

### Bug hunt: Extras that vanished

The extras are under `[project.optional_dependencies]`, with an underscore, so tools ignore them: the standard key is `optional-dependencies`, with a hyphen, and this program (like pip) finds no extras and prints `[]`. Fix the table name so it prints `['dev', 'pdf']`.

In [ ]:
import tomllib

text = '''
[project]
name = "textkit"
version = "1.0.0"

[project.optional_dependencies]
dev = ["pytest"]
pdf = ["pypdf"]
'''
project = tomllib.loads(text)["project"]
print(sorted(project.get("optional-dependencies", {})))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the standard key\"\n    expect_output(\"['dev', 'pdf']\")")

### Bug hunt: Dependencies in the wrong table

The runtime dependencies were put under `[build-system] requires`, which lists only what's needed **to build** the project, so installing it wouldn't install them, and this prints `[]`. Move `rich` and `regex` to `dependencies` in `[project]`, leaving `requires = ["hatchling"]`, so this prints `['rich', 'regex'] ['hatchling']`.

In [ ]:
import tomllib

text = '''
[build-system]
requires = ["hatchling", "rich", "regex"]
build-backend = "hatchling.build"

[project]
name = "textkit"
version = "1.0.0"
'''
data = tomllib.loads(text)
print(data["project"].get("dependencies", []), data["build-system"]["requires"] if data["project"].get("dependencies") else "")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"runtime and build requirements\"\n    expect_output(\"['rich', 'regex'] ['hatchling']\")")

### Puzzle: A minimal pyproject.toml

Arrange the lines of a minimal `pyproject.toml` for a project built with hatchling, with one dependency and one command. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the normalised form of "My.Cool_Package"
q = ___   # the table that lists what's needed to build, not run: "build-system" or "project"
r = ___   # the extras in "textkit[pdf,dev]>=1.0", sorted, as a list
s = ___   # True or False: tomllib.load needs a file opened in binary mode
t = ___   # what "pip install textkit[pdf]" adds to textkit's own dependencies: the "pdf" what?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"my-cool-package\" and q == \"build-system\" and r == [\"dev\", \"pdf\"], \"p, q and r\"\n    assert s is True and t in (\"extra\", \"optional dependencies\", \"optional-dependencies\"), \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a pyproject.toml checker

Write `check(text, files)`: return a list of problem messages (in the order of the rules below) for a `pyproject.toml`, where `files` is the set of paths in the project. If the text isn't valid TOML, return just `["not valid TOML: ..."]` (any message after the colon). Otherwise check:
1. `"no [build-system]"` if there's no `build-system` table; otherwise `"build-system needs requires"` and `"build-system needs build-backend"` if either is missing.
2. `"no [project]"` if there's no `project` table (and stop checking there).
3. `"invalid name: NAME"` if the name is missing (written as `invalid name: None`) or invalid.
4. `"no version"` if there's no version and it isn't dynamic.
5. `"readme not found: PATH"` if `readme` is set to a path that isn't in `files`.
6. `"bad dependency: TEXT"` for each dependency (and each optional dependency, extras in order) that `parse_requirement` (from the practice, given in the starter) rejects.
7. `"bad script: NAME"` for each console script whose value isn't `module:function`.
8. `"requires-python looks wrong: TEXT"` if `requires-python` doesn't start with an operator from `>=`, `>`, `<=`, `<`, `==`, `!=` and `~=`.

In [ ]:
import re, tomllib

OPERATORS = ["===", "==", "!=", "<=", ">=", "~=", "<", ">"]

def parse_requirement(text):
    spec, _, marker = text.partition(";")
    m = re.fullmatch(r"\s*([A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?)\s*(\[([^\]]*)\])?\s*(.*?)\s*", spec)
    if not m:
        raise ValueError(f"invalid requirement: {text!r}")
    name, _, extras_text, rest = m.groups()
    specifiers = []
    if rest:
        for clause in rest.split(","):
            clause = clause.strip()
            op = next((o for o in OPERATORS if clause.startswith(o)), None)
            version = clause[len(op):].strip() if op else ""
            if op is None or not version or not re.fullmatch(r"[A-Za-z0-9.*+!_-]+", version):
                raise ValueError(f"invalid version clause: {clause!r}")
            specifiers.append((op, version))
    return {"name": name, "specifiers": specifiers}

def check(text, files):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checker():\n    \"every rule\"\n    good = '[build-system]\\nrequires = [\"hatchling\"]\\nbuild-backend = \"hatchling.build\"\\n\\n[project]\\nname = \"textkit\"\\nversion = \"1.0\"\\nreadme = \"README.md\"\\nrequires-python = \">=3.10\"\\ndependencies = [\"rich>=13\"]\\n\\n[project.scripts]\\ntk = \"textkit.cli:main\"\\n'\n    assert check(good, {\"README.md\", \"pyproject.toml\"}) == []\n    bad = '[build-system]\\nrequires = [\"hatchling\"]\\n\\n[project]\\nname = \"-textkit\"\\nreadme = \"docs/README.md\"\\nrequires-python = \"3.10+\"\\ndependencies = [\"rich >> 13\", \"regex\"]\\n\\n[project.optional-dependencies]\\ndev = [\"pytest\", \"mypy==\"]\\n\\n[project.scripts]\\ntk = \"textkit.cli\"\\n'\n    assert check(bad, {\"README.md\"}) == [\n        \"build-system needs build-backend\",\n        \"invalid name: -textkit\",\n        \"no version\",\n        \"readme not found: docs/README.md\",\n        \"bad dependency: rich >> 13\",\n        \"bad dependency: mypy==\",\n        \"bad script: tk\",\n        \"requires-python looks wrong: 3.10+\",\n    ]\n    assert check('[tool.x]\\na = 1\\n', set()) == [\"no [build-system]\", \"no [project]\"]\n    assert check('[project]\\nversion = 1.0.0\\n', set())[0].startswith(\"not valid TOML: \")\n    assert check('[build-system]\\nrequires=[]\\nbuild-backend=\"b\"\\n[project]\\ndynamic=[\"version\"]\\n', set()) == [\"invalid name: None\"]")

## Warm-up

A TOML file becomes plain Python dictionaries and lists. Everything a packaging tool knows about your project comes from reading a structure like this one.

## TOML

**TOML** ("Tom's Obvious Minimal Language") is a configuration format designed to be easy to read and to map unambiguously onto dictionaries:

- `key = value` pairs. Strings are **quoted** (`"double"` or `'literal'`, where backslashes aren't escapes), numbers and `true` / `false` aren't. Comments start with `#`.
- `[table]` starts a table (a dictionary); `[project.urls]` is the table `urls` inside `project`. Dotted keys do the same: `project.name = "x"`.
- Arrays are `[1, 2, 3]` and can span lines, with a trailing comma allowed. Inline tables are `{ name = "Ada", email = "ada@example.com" }`.
- `[[array.of.tables]]` appends a table to a list each time it appears.
- Keys can't be defined twice, and values have types: `version = 1.0.0` isn't a string (or valid TOML at all) until it's quoted.

Python reads TOML with **`tomllib`** (in the standard library since 3.11): `tomllib.loads(text)` or `tomllib.load(file)` with a file opened in **binary** mode (`"rb"`). It only reads; to write TOML you need a third-party library such as `tomli-w`, or (for simple cases) your own few lines, as in today's practice.

## build-system

A source tree doesn't install itself. **Build backends** turn it into installable files (Day 3), and the `[build-system]` table says which one to use (this comes from PEPs 517 and 518):

```toml
[build-system]
requires = ["hatchling>=1.24"]
build-backend = "hatchling.build"
```

- `requires` lists the packages needed **to build** (not to run) the project. pip installs them into a temporary, isolated environment first.
- `build-backend` names the object that does the building. Common choices: `setuptools.build_meta` (setuptools, the oldest and most flexible), `hatchling.build` (Hatch), `flit_core.buildapi` (Flit, for pure-Python packages), `poetry.core.masonry.api` (Poetry), `pdm.backend` (PDM).

The backend matters much less than it used to: for a pure-Python package in the src layout, any of them works with almost the same `pyproject.toml`. Old projects have a `setup.py` script instead; it still works, but new projects don't need one.

## The project table

`[project]` holds the **metadata** every backend understands (PEP 621):

```toml
[project]
name = "textkit"
version = "0.3.0"
description = "Small, fast text utilities."
readme = "README.md"
requires-python = ">=3.10"
license = "MIT"
authors = [{ name = "Ada Lovelace", email = "ada@example.com" }]
classifiers = [
    "Programming Language :: Python :: 3",
    "Operating System :: OS Independent",
]
dependencies = [
    "regex>=2024.1",
    "rich>=13,<15",
]

[project.optional-dependencies]
dev = ["pytest>=8", "mypy>=1.10"]
pdf = ["pypdf>=4"]

[project.scripts]
textkit = "textkit.cli:main"

[project.urls]
Homepage = "https://example.com/textkit"
Issues = "https://example.com/textkit/issues"

[tool.pytest.ini_options]
testpaths = ["tests"]
```

- `name` and `version` are required, unless a field is listed in **`dynamic`** (`dynamic = ["version"]`), meaning the backend works it out, for instance from `__version__` in the code or from a Git tag.
- `requires-python` stops pip installing the project on Python versions it doesn't support.
- `dependencies` are what the project needs **to run**, as requirement strings (below). Keep them as loose as is safe (Day 5).
- `optional-dependencies` are named groups of **extras**: `pip install textkit[pdf]` adds `pypdf`. A `dev` or `test` extra is a common (if debated) way to list development tools.
- `scripts` are **console scripts**: installing creates a `textkit` command that imports `textkit.cli` and calls `main()`, exiting with its return value. `gui-scripts` is the same for GUI programs, and `[project.entry-points."group"]` registers **plugins** that other programs discover by group name.
- `[tool.NAME]` tables belong to individual tools: `[tool.pytest.ini_options]`, `[tool.ruff]`, `[tool.mypy]` and so on.

## Names and requirements

**Names.** A project name may contain letters, digits, `.`, `_` and `-`, and must start and end with a letter or digit. Names are compared after **normalisation** (PEP 503): lower case, with every run of `-`, `_` and `.` replaced by a single `-`. So `Text_Kit`, `text.kit` and `TEXT--kit` are all the same project, `text-kit`, and you can't publish one if another exists.

**Requirement strings** (PEP 508) say which project, which version, which extras and under which conditions:

```text
requests
rich>=13,<15
textkit[pdf,dev]>=0.3
tomli>=2; python_version < "3.11"
pywin32>=306; sys_platform == "win32"
```

- the **name**, optionally followed by **extras** in square brackets;
- a **version specifier**: comma-separated clauses such as `>=13`, `<15`, `==2.31.*`, `!=1.4.2` and `~=1.4` (compatible release, Day 5), all of which must hold;
- after a `;`, an **environment marker**: a condition on the Python version, the platform and so on, so one requirement can apply only where it's needed. The `tomli` line above is the classic: Python 3.11 and later have `tomllib` built in.

## Practice

## On your own computer

Build-tool front ends read the same file. Two worth trying on the src-layout project from yesterday:

```bash
pip install -e ".[dev]"
```

```bash
uv init --package demo && cd demo && cat pyproject.toml
```

The first installs your project in editable mode **with** its `dev` extra (the quotes stop your shell expanding the brackets). The second uses **uv**, a fast all-in-one tool for environments, dependencies and builds, to create a new project; read the `pyproject.toml` it writes and you'll recognise every line.

## Quiz

## Daily challenge

## Before you go

Write the `pyproject.toml` for a project of your own from earlier in the course: name, version, Python versions, dependencies and a command. Tomorrow: **publishing a package**: what a wheel really is, how one is built, and how it gets to the Package Index.